# Memori RAG Evaluation — LoCoMo Benchmark

This notebook runs a full end-to-end evaluation of **Memori's retrieval pipeline** on the [LoCoMo](https://github.com/snap-research/locomo) long-conversation benchmark.

### How it works

1. **Retrieve** — For each benchmark question, a hybrid search (FAISS semantic search + BM25 keyword matching) finds the most relevant *memories and summaries* from the pre-built indexes.
2. **Answer** — An LLM generates an answer using **only** the retrieved context (no raw chat transcripts).
3. **Judge** — A separate LLM call compares the generated answer to the gold standard and labels it `CORRECT` or `WRONG`.
4. **Report** — Accuracy metrics are computed per category and per conversation.

### Prerequisites

- **`indexes_gemma/`** must exist (run [`01_load_indexes.ipynb`](01_load_indexes.ipynb) first).
- **`OPENAI_API_KEY`** must be set in `.env` (used for answer generation and judging).
- See the [README](README.md) for full setup instructions.

### Data source

Benchmark questions come from [`locomo10.json`](https://raw.githubusercontent.com/lborro/memori-aa-data/refs/heads/main/locomo10.json) (fetched automatically). To use a local copy, set `LOCOMO_LOCAL_PATH` in the configuration cell below.

### Pipeline overview — run sections in order

| Section | Stage | What it does |
|---------|-------|--------------|
| **1** | Load benchmark data | Fetch the LoCoMo JSON; parse sessions and QA pairs |
| **2** | Load indexes | Initialise the embedding model; load FAISS + BM25 indexes |
| **3** | Sanity check | Run one test query to verify the search stack works |
| **4** | Generate answers | Embed questions, retrieve context, call the LLM |
| **5** | Judge answers | LLM-as-judge scores each answer as CORRECT or WRONG |
| **6** | Compute metrics | Print accuracy tables; save results to `results_gemma/` |
| **7** | Inspect wrong answers | Browse failures to diagnose retrieval or prompt issues |
| **8** | Token analysis | Compare RAG context size vs a full-session baseline |


In [ ]:
import asyncio
import json
import logging
import os
import re
import time
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

os.environ.setdefault("OMP_NUM_THREADS", "1")

import faiss
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import AsyncOpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
)
from tqdm.asyncio import tqdm as atqdm

load_dotenv()

# ── LLM configuration ────────────────────────────────────────────────
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-mini")  # used for answering and judging

# ── Embedding model (must match the one used to build indexes) ───────
EMBED_MODEL_NAME = "google/embeddinggemma-300m"

# ── Data paths ───────────────────────────────────────────────────────
LOCOMO_URL = "https://raw.githubusercontent.com/lborro/memori-aa-data/refs/heads/main/locomo10.json"
LOCOMO_LOCAL_PATH = Path(os.getenv("LOCOMO_LOCAL_PATH", ""))  # set to use a local copy
INDEX_DIR = Path("./indexes_gemma")       # FAISS indexes from notebook 01
RESULTS_DIR = Path("./results_gemma")     # evaluation output
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Concurrency (max parallel LLM calls) ─────────────────────────────
SEM_LLM = asyncio.Semaphore(8)

# ── Retrieval settings ───────────────────────────────────────────────
TOP_K = 10            # number of documents to retrieve per question
RRF_K = 60            # RRF smoothing constant (higher = less weight on top ranks)
EVAL_CATEGORIES = [1, 2, 3, 4]  # 1=Multi-hop, 2=Temporal, 3=Open-domain, 4=Single-hop

# ── Logging ──────────────────────────────────────────────────────────
logging.basicConfig(level=logging.WARNING, format="%(levelname)s | %(message)s")
logger = logging.getLogger("rag_eval_gemma")
logger.setLevel(logging.INFO)

openai_client = AsyncOpenAI(api_key=OPENAI_API_KEY)

In [ ]:
ANSWER_PROMPT = """
    You are an intelligent memory assistant tasked with retrieving accurate information from conversation memories.

    # CONTEXT:
    You have access to two types of information derived conversation:
    - Memories: timestamped factual triples extracted from conversations.
    - Summaries: high-level conversation summaries (also timestamped) that
      provide broader context around the memories.
    Together they form the only evidence you may use to answer the benchmark question.

    # INSTRUCTIONS:
    1. Carefully analyze all provided memories and summaries
    2. Pay special attention to the timestamps to determine the answer
    3. If the question asks about a specific event or fact, look for direct evidence in the memories
    4. If the memories contain contradictory information, prioritize the most recent memory
    5. If there is a question about time references (like "last year", "two months ago", etc.),
       calculate the actual date based on the memory timestamp. For example, if a memory from
       4 May 2022 mentions "went to India last year," then the trip occurred in 2021.
    6. Always convert relative time references to specific dates, months, or years. For example,
       convert "last year" to "2022" or "two months ago" to "March 2023" based on the memory
       timestamp. Ignore the reference while answering the question.
    7. Focus only on the content of the memories. Do not confuse character
       names mentioned in memories with the actual users who created those memories.
    8. The answer should be less than 5-6 words.

    # APPROACH (Think step by step):
    1. First, examine all memories that contain information related to the question
    2. Use summaries for broader context when memories alone are insufficient
    3. Examine the timestamps and content carefully
    4. Look for explicit mentions of dates, times, locations, or events that answer the question
    5. If the answer requires calculation (e.g., converting relative time references), show your work
    6. Formulate a precise, concise answer based solely on the evidence in the memories
    7. Double-check that your answer directly addresses the question asked
    8. Ensure your final answer is specific and avoids vague time references

    {{memories}}

    Question: {{question}}
    Answer:
    """

ACCURACY_PROMPT = """
Your task is to label an answer to a question as 'CORRECT' or 'WRONG'. You will be given the following data:
    (1) a question (posed by one user to another user),
    (2) a 'gold' (ground truth) answer,
    (3) a generated answer
which you will score as CORRECT/WRONG.

The point of the question is to ask about something one user should know about the other user based on their prior conversations.
The gold answer will usually be a concise and short answer that includes the referenced topic, for example:
Question: Do you remember what I got the last time I went to Hawaii?
Gold answer: A shell necklace
The generated answer might be much longer, but you should be generous with your grading - as long as it touches on the same topic as the gold answer, it should be counted as CORRECT.

For time related questions, the gold answer will be a specific date, month, year, etc. The generated answer might be much longer or use relative time references (like "last Tuesday" or "next month"), but you should be generous with your grading - as long as it refers to the same date or time period as the gold answer, it should be counted as CORRECT. Even if the format differs (e.g., "May 7th" vs "7 May"), consider it CORRECT if it's the same date.

Now it's time for the real question:
Question: {question}
Gold answer: {gold_answer}
Generated answer: {generated_answer}

First, provide a short (one sentence) explanation of your reasoning, then finish with CORRECT or WRONG.
Do NOT include both CORRECT and WRONG in your response, or it will break the evaluation script.

Just return the label CORRECT or WRONG in a json format with the key as "label".
"""

## 1 · Load benchmark data

Downloads (or reads locally) the LoCoMo benchmark JSON and extracts:
- **Sessions** — the raw conversation turns, used later only for the token-count baseline (section 8).
- **Questions + gold answers** — the QA pairs we will evaluate against.

Each question belongs to one of five categories: *Single-hop*, *Multi-hop*, *Temporal*, *Open-domain*, or *Adversarial*. By default, categories 1-4 are evaluated (configured via `EVAL_CATEGORIES` above).


In [ ]:
def load_locomo() -> list[dict]:
    """Load locomo10.json from LOCOMO_URL, or from LOCOMO_LOCAL_PATH if that file exists."""
    if LOCOMO_LOCAL_PATH.is_file():
        with open(LOCOMO_LOCAL_PATH, encoding="utf-8") as f:
            return json.load(f)
    req = urllib.request.Request(
        LOCOMO_URL, headers={"User-Agent": "Memori-benchmark/1.0"}
    )
    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read().decode("utf-8"))


def extract_sessions(conversation: dict) -> list[dict]:
    """Parse ordered sessions, formatting each as a single user message with speaker names."""
    conv_data = conversation["conversation"]
    speaker_a = conv_data.get("speaker_a", "Speaker A")
    speaker_b = conv_data.get("speaker_b", "Speaker B")

    session_keys = sorted(
        [
            k
            for k in conv_data
            if k.startswith("session_") and not k.endswith("_date_time")
        ],
        key=lambda k: int(k.split("_")[1]),
    )

    sessions = []
    for skey in session_keys:
        ts_key = f"{skey}_date_time"
        turns = conv_data[skey]
        dialog_lines = []
        for turn in turns:
            speaker = turn.get("speaker", "Unknown")
            content = turn.get("text", "")
            blip = turn.get("blip_caption", "")
            query = turn.get("query", "")
            if blip or query:
                img_desc = query if query else blip
                if query and blip:
                    img_desc = f"{query} — {blip}"
                content = f"[Shared image: {img_desc}] {content}".strip()
            dialog_lines.append(f"{speaker}: {content}")

        session_text = "\n".join(dialog_lines)
        messages = [{"role": "user", "content": session_text}]

        sessions.append(
            {
                "session_key": skey,
                "timestamp": conv_data.get(ts_key, ""),
                "messages": messages,
                "speaker_a": speaker_a,
                "speaker_b": speaker_b,
            }
        )
    return sessions


def extract_questions(conversation: dict) -> list[dict]:
    """Extract QA pairs; category 5 uses adversarial_answer (unanswerable)."""
    questions = []
    for qa in conversation.get("qa", []):
        q = {
            "question": qa["question"],
            "category": qa["category"],
            "evidence": qa.get("evidence", []),
        }
        if qa["category"] == 5:
            q["gold_answer"] = qa.get("adversarial_answer", "unanswerable")
        else:
            q["gold_answer"] = qa.get("answer", "")
        questions.append(q)
    return questions


# ── Load & summarise ─────────────────────────────────────────────────
locomo_data = load_locomo()
_data_src = (
    str(LOCOMO_LOCAL_PATH.resolve()) if LOCOMO_LOCAL_PATH.is_file() else LOCOMO_URL
)
print(f"LoCoMo source: {_data_src}\n")

conversations: list[dict] = []
for conv in locomo_data:
    sessions = extract_sessions(conv)
    questions = extract_questions(conv)
    conversations.append(
        {
            "conv_id": conv["sample_id"],
            "sessions": sessions,
            "questions": [q for q in questions if q["category"] in EVAL_CATEGORIES],
        }
    )

total_sessions = sum(len(c["sessions"]) for c in conversations)
total_questions = sum(len(c["questions"]) for c in conversations)
print(
    f"Loaded {len(conversations)} conversations | {total_sessions} sessions | {total_questions} questions\n"
)

CATEGORY_NAMES = {
    1: "Multi-hop",
    2: "Temporal",
    3: "Open-domain",
    4: "Single-hop",
    5: "Adversarial",
}

for c in conversations:
    q_by_cat: dict[int, int] = {}
    for q in c["questions"]:
        q_by_cat[q["category"]] = q_by_cat.get(q["category"], 0) + 1
    cats = "  ".join(
        f"{CATEGORY_NAMES.get(k, '?')}:{v}" for k, v in sorted(q_by_cat.items())
    )
    print(
        f"  {c['conv_id']:>8}: {len(c['sessions']):>2} sessions | {len(c['questions']):>3} questions | {cats}"
    )

## 2 · Load the embedding model and search indexes

This section loads two things:

1. **The embedding model** (`google/embeddinggemma-300m`) — the same model used in notebook 01 to build the indexes. It will be used here to embed each benchmark question before searching.
2. **The hybrid indexes** — for each conversation, a FAISS vector index (semantic search) and a BM25 index (keyword search) are loaded from `indexes_gemma/`. At query time, results from both are merged via [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) (RRF).

The indexed documents are **memories and summaries** produced by Memori's Advanced Augmentation pipeline.

In [ ]:
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
embedding_dim = embed_model.get_sentence_embedding_dimension()
print(f"Embedding model: {EMBED_MODEL_NAME}")
print(f"Embedding dimension: {embedding_dim}")

In [ ]:
class PipelineErrors:
    """Accumulates errors without crashing the pipeline."""

    def __init__(self):
        self.errors: list[dict] = []

    def log(self, stage: str, item_id: str, error: Exception):
        self.errors.append(
            {"stage": stage, "item_id": str(item_id)[:120], "error": str(error)[:200]}
        )
        logger.warning(f"[{stage}] {item_id}: {error}")

    @property
    def count(self) -> int:
        return len(self.errors)


pipeline_errors = PipelineErrors()


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=2, min=3, max=60),
)
async def call_llm(prompt: str, system: str | None = None) -> str:
    messages: list[dict] = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})
    async with SEM_LLM:
        resp = await openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=messages,
            temperature=0,
        )
        return resp.choices[0].message.content.strip()

In [ ]:
@dataclass
class HybridIndex:
    """FAISS (semantic) + BM25 (keyword) hybrid index with Reciprocal Rank Fusion."""

    conv_id: str
    documents: list[dict] = field(default_factory=list)
    faiss_index: Any = None
    bm25: Any = None
    _tok_corpus: list[list[str]] = field(default_factory=list)

    def search(
        self, query_embedding: np.ndarray, query_text: str, top_k: int = TOP_K
    ) -> list[dict]:
        if not self.documents:
            return []

        k = min(top_k * 2, len(self.documents))

        qe = query_embedding.copy().reshape(1, -1)
        faiss.normalize_L2(qe)
        _, idx_sem = self.faiss_index.search(qe, k)
        sem_ranks = {int(i): r for r, i in enumerate(idx_sem[0]) if i >= 0}

        bm25_scores = self.bm25.get_scores(self._tokenize(query_text))
        bm25_top = np.argsort(bm25_scores)[::-1][:k]
        bm25_ranks = {int(i): r for r, i in enumerate(bm25_top)}

        all_idx = set(sem_ranks) | set(bm25_ranks)
        rrf: dict[int, float] = {}
        for i in all_idx:
            score = 0.0
            if i in sem_ranks:
                score += 1.0 / (RRF_K + sem_ranks[i] + 1)
            if i in bm25_ranks:
                score += 1.0 / (RRF_K + bm25_ranks[i] + 1)
            rrf[i] = score

        ranked = sorted(rrf, key=lambda x: rrf[x], reverse=True)[:top_k]
        return [self.documents[i] for i in ranked]

    @classmethod
    def load(cls, directory: Path) -> "HybridIndex":
        with open(directory / "metadata.json") as f:
            meta = json.load(f)
        if isinstance(meta, list):
            conv_id = directory.name
            documents = meta
        else:
            conv_id = meta["conv_id"]
            documents = meta["documents"]

        idx = cls(conv_id=conv_id)
        idx.documents = documents

        faiss_path = directory / "faiss.index"
        if not faiss_path.exists():
            faiss_path = directory / "index.faiss"
        idx.faiss_index = faiss.read_index(str(faiss_path))

        idx._tok_corpus = [idx._tokenize(idx._doc_text(t)) for t in idx.documents]
        if idx._tok_corpus:
            idx.bm25 = BM25Okapi(idx._tok_corpus)

        return idx

    @staticmethod
    def _tokenize(text: str) -> list[str]:
        return text.lower().split()

    @staticmethod
    def _doc_text(triple: dict) -> str:
        content = triple.get("content", "")
        context = triple.get("context", "")
        if isinstance(context, dict):
            context = " ".join(str(v) for v in context.values())
        return f"{content} {context}".strip()

In [ ]:
t0 = time.perf_counter()

indexes: dict[str, HybridIndex] = {}
for conv_dir in sorted(INDEX_DIR.iterdir()):
    if not conv_dir.is_dir():
        continue
    try:
        idx = HybridIndex.load(conv_dir)
        indexes[idx.conv_id] = idx
        logger.info(
            f"  {idx.conv_id}: {len(idx.documents):>4} documents, dim={idx.faiss_index.d}"
        )
    except Exception as e:
        pipeline_errors.log("index_load", conv_dir.name, e)

elapsed = time.perf_counter() - t0
print(f"\nLoaded {len(indexes)} indexes from {INDEX_DIR} in {elapsed:.1f}s")
for cid, idx in indexes.items():
    print(f"  {cid:>8}: {len(idx.documents):>4} documents (dim={idx.faiss_index.d})")

## 3 · Sanity check

Before running the full evaluation, test a single query against one conversation's index to make sure everything is wired up correctly. The top results should be semantically relevant to the query.

**If the results look wrong or empty:** check that `INDEX_DIR` points to the correct directory, or re-run notebook 01 to rebuild the indexes.

In [ ]:
QUERY = "What is Caroline's relationship status?"
CONV_ID = "conv-26"

index = indexes[CONV_ID]
q_emb = embed_model.encode_query(QUERY).astype(np.float32)

results = index.search(q_emb, QUERY, top_k=5)

print(f"Query:  {QUERY}")
print(f"Index:  {CONV_ID}  ({len(index.documents)} documents)")
print(f"Top-{len(results)} results:\n")
for i, doc in enumerate(results, 1):
    content = doc.get("content", "")
    context = doc.get("context", "")
    session = doc.get("_session", "")
    print(f"  {i}. [{session}] {content}")
    if context:
        print(f"     context: {context}")
    print()

## 4 · Generate answers (parallel)

This is the core retrieval-augmented generation step. For every benchmark question:

1. **Embed the question** using EmbeddingGemma (same model weights as indexing).
2. **Retrieve** the top-K most relevant documents via hybrid search (FAISS + BM25 with RRF).
3. **Format** the retrieved memories and summaries into a context block.
4. **Call the LLM** with the context and question to produce an answer.

All questions are processed concurrently (with a concurrency limiter to stay within API rate limits). The LLM sees **only** the retrieved context — no additional evidence.

In [ ]:
def _collect_summaries(results: list[dict]) -> list[tuple[str, str, str]]:
    """Collect unique (session, timestamp, summary) tuples across retrieved triples."""
    seen: set[str] = set()
    summaries: list[tuple[str, str, str]] = []
    for doc in results:
        session = doc.get("_session", "")
        timestamp = doc.get("_timestamp", "")
        for s in (
            [doc["_summary"]] if doc.get("_summary") else doc.get("_summaries", [])
        ):
            if s and s not in seen:
                seen.add(s)
                summaries.append((session, timestamp, s))
    return summaries


def format_memories(results: list[dict]) -> str:
    """Format retrieved context into Memories + Summaries sections."""
    sections: list[str] = []

    lines = []
    for i, doc in enumerate(results, 1):
        content = doc.get("content", "")
        context = doc.get("context", "")
        timestamp = doc.get("_timestamp", "")
        mem = f"  {i}."
        if timestamp:
            mem += f" [{timestamp}]"
        mem += f" {content} ({context})"
        lines.append(mem)

    if lines:
        sections.append("Memories:\n" + "\n".join(lines))

    summaries = _collect_summaries(results)
    if summaries:
        summary_lines = []
        for sess, ts, s in summaries:
            tag = f"{sess} | {ts}" if ts else sess
            summary_lines.append(f"  - [{tag}] {s}" if tag else f"  - {s}")
        sections.append("Summaries:\n" + "\n".join(summary_lines))

    return "\n\n".join(sections)


async def generate_answer(
    question: dict, index: HybridIndex, q_emb: np.ndarray
) -> dict:
    """Retrieve context via hybrid search and generate an LLM answer."""
    q_text = question["question"]

    try:
        results = index.search(q_emb, q_text)
        memories_text = format_memories(results)

        prompt = ANSWER_PROMPT.replace("{{memories}}", memories_text).replace(
            "{{question}}", q_text
        )
        answer = await call_llm(prompt)
    except Exception as e:
        pipeline_errors.log("answer_generation", q_text[:60], e)
        answer = f"ERROR: {e}"
        results = []
        memories_text = ""

    return {
        **question,
        "generated_answer": answer,
        "retrieved_count": len(results),
        "memories_text": memories_text,
        "retrieved_context": [
            {"content": r.get("content", ""), "context": r.get("context", "")}
            for r in results
        ],
        "conv_id": index.conv_id,
    }

In [ ]:
async def generate_all_answers() -> list[dict]:
    """Embed all questions locally, then generate LLM answers in parallel."""
    all_questions: list[tuple[dict, HybridIndex]] = []
    for conv in conversations:
        index = indexes.get(conv["conv_id"])
        if index is None:
            for q in conv["questions"]:
                pipeline_errors.log(
                    "answer_generation",
                    f"{conv['conv_id']}/{q['question'][:40]}",
                    Exception("No index available"),
                )
            continue
        for q in conv["questions"]:
            all_questions.append((q, index))

    q_texts = [q["question"] for q, _ in all_questions]
    print(f"Embedding {len(q_texts)} questions with {EMBED_MODEL_NAME}...")
    q_embeddings = embed_model.encode_query(
        q_texts, batch_size=64, show_progress_bar=True
    )
    q_embeddings = np.array(q_embeddings, dtype=np.float32)
    print(f"Embeddings shape: {q_embeddings.shape}")

    tasks = [
        generate_answer(q, idx, q_emb)
        for (q, idx), q_emb in zip(all_questions, q_embeddings, strict=True)
    ]
    return await atqdm.gather(*tasks, desc="Generating answers")


t0 = time.perf_counter()
answer_results = await generate_all_answers()
elapsed = time.perf_counter() - t0

print(f"\n  Generated {len(answer_results)} answers in {elapsed:.1f}s")
print(f"  Pipeline errors so far: {pipeline_errors.count}")

## 5 · Evaluate answers (LLM-as-judge)

Each generated answer is compared to the gold (ground-truth) answer by a second LLM call acting as a judge. The judge is instructed to be lenient — it marks an answer `CORRECT` as long as it refers to the same topic, date, or fact as the gold answer, even if the wording or format differs.

The judge returns a JSON object with a `label` field (`CORRECT` or `WRONG`). A fallback regex parser handles cases where the response isn't valid JSON.

In [ ]:
def parse_judge_label(response: str) -> str | None:
    """Extract CORRECT/WRONG from the judge response, handling markdown fences."""
    cleaned = re.sub(r"```(?:json)?\s*", "", response)
    cleaned = re.sub(r"```", "", cleaned).strip()

    try:
        parsed = json.loads(cleaned)
        label = str(parsed.get("label", "")).upper()
        if label in ("CORRECT", "WRONG"):
            return label
    except (json.JSONDecodeError, AttributeError):
        pass

    upper = response.upper()
    has_correct = "CORRECT" in upper
    has_wrong = "WRONG" in upper
    if has_correct and not has_wrong:
        return "CORRECT"
    if has_wrong and not has_correct:
        return "WRONG"

    return None


async def evaluate_answer(result: dict) -> dict:
    """Run the LLM judge on a single answer."""
    prompt = ACCURACY_PROMPT.format(
        question=result["question"],
        gold_answer=result["gold_answer"],
        generated_answer=result["generated_answer"],
    )

    label = None
    response = ""
    try:
        response = await call_llm(prompt)
        label = parse_judge_label(response)
        if label is None:
            pipeline_errors.log(
                "eval_parse",
                result["question"][:60],
                Exception(f"Unparseable response: {response[:120]}"),
            )
    except Exception as e:
        pipeline_errors.log("evaluation", result["question"][:60], e)
        response = f"ERROR: {e}"

    return {**result, "judge_response": response, "label": label}


async def evaluate_all_answers() -> list[dict]:
    tasks = [evaluate_answer(r) for r in answer_results]
    return await atqdm.gather(*tasks, desc="Evaluating answers")


t0 = time.perf_counter()
eval_results = await evaluate_all_answers()
elapsed = time.perf_counter() - t0

print(f"\n  Evaluated {len(eval_results)} answers in {elapsed:.1f}s")
print(f"  Total pipeline errors: {pipeline_errors.count}")

## 6 · Performance metrics

Compute and display overall accuracy, broken down by:
- **Question category** (Single-hop, Multi-hop, Temporal, Open-domain).
- **Conversation** (to spot per-conversation strengths or weaknesses).

Results are also saved as a timestamped JSON file under `results_gemma/` for later comparison across runs.

> Remember: all answers were generated using **only** retrieved memories and summaries — not full conversation transcripts.

In [ ]:
from datetime import datetime, timezone

df = pd.DataFrame(eval_results)

evaluated = df[df["label"].notna()]
failed = df[df["label"].isna()]

total = len(df)
n_evaluated = len(evaluated)
n_correct = int((evaluated["label"] == "CORRECT").sum()) if n_evaluated else 0
n_wrong = int((evaluated["label"] == "WRONG").sum()) if n_evaluated else 0
n_failed = len(failed)

accuracy = n_correct / n_evaluated if n_evaluated > 0 else 0
failure_rate = n_failed / total if total > 0 else 0

print("=" * 62)
print("  MEMORI RAG EVALUATION – LOCOMO (EmbeddingGemma-300M)")
print("=" * 62)
print(f"  Embedding model:         {EMBED_MODEL_NAME}")
print(f"  LLM model:               {OPENAI_MODEL}")
print(f"  Total questions:         {total:>6}")
print(f"  Successfully evaluated:  {n_evaluated:>6}")
print(f"    Correct:               {n_correct:>6}")
print(f"    Wrong:                 {n_wrong:>6}")
print(f"  Failed (parse/API):      {n_failed:>6}")
print("-" * 62)
print(f"  Accuracy:                {accuracy:>6.1%}")
print(f"  Failure rate:            {failure_rate:>6.1%}")
print("=" * 62)

# ── Per-category breakdown ───────────────────────────────────────────
print("\nAccuracy by Question Category\n")
cat_agg = (
    evaluated.groupby("category")
    .agg(total=("label", "count"), correct=("label", lambda s: (s == "CORRECT").sum()))
    .reset_index()
)
cat_agg["accuracy"] = (cat_agg["correct"] / cat_agg["total"]).map("{:.1%}".format)
cat_agg["category_name"] = cat_agg["category"].map(CATEGORY_NAMES)
print(
    cat_agg[["category", "category_name", "total", "correct", "accuracy"]].to_string(
        index=False
    )
)

# ── Per-conversation breakdown ───────────────────────────────────────
print("\nAccuracy by Conversation\n")
conv_agg = (
    evaluated.groupby("conv_id")
    .agg(total=("label", "count"), correct=("label", lambda s: (s == "CORRECT").sum()))
    .reset_index()
)
conv_agg["accuracy"] = (conv_agg["correct"] / conv_agg["total"]).map("{:.1%}".format)
print(conv_agg.to_string(index=False))

# ── Error log ────────────────────────────────────────────────────────
if pipeline_errors.errors:
    print(f"\nError Log ({pipeline_errors.count} total errors)\n")
    err_df = pd.DataFrame(pipeline_errors.errors)
    print(err_df.groupby("stage").size().rename("count").to_string())

# ── Save results ─────────────────────────────────────────────────────
run_ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

per_category = {
    row["category_name"]: {
        "total": int(row["total"]),
        "correct": int(row["correct"]),
        "accuracy": row["accuracy"],
    }
    for _, row in cat_agg.iterrows()
}
per_conversation = {
    row["conv_id"]: {
        "total": int(row["total"]),
        "correct": int(row["correct"]),
        "accuracy": row["accuracy"],
    }
    for _, row in conv_agg.iterrows()
}

save_payload = {
    "run_timestamp": run_ts,
    "config": {
        "embed_model": EMBED_MODEL_NAME,
        "llm_model": OPENAI_MODEL,
        "top_k": TOP_K,
        "rrf_k": RRF_K,
        "index_dir": str(INDEX_DIR),
    },
    "summary": {
        "total": total,
        "evaluated": n_evaluated,
        "correct": n_correct,
        "wrong": n_wrong,
        "failed": n_failed,
        "accuracy": round(accuracy, 4),
        "failure_rate": round(failure_rate, 4),
        "by_category": per_category,
        "by_conversation": per_conversation,
    },
    "details": eval_results,
    "errors": pipeline_errors.errors,
}

results_path = RESULTS_DIR / f"eval_{run_ts}.json"
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(save_payload, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to {results_path}")

## 7 · Inspect wrong answers

Browse the questions the model got wrong. This is helpful for diagnosing:
- **Retrieval gaps** — the relevant memory wasn't in the top-K results.
- **Prompt issues** — the LLM misinterpreted the context or the question.
- **Temporal reasoning failures** — the model couldn't resolve relative time references.

The first 10 wrong answers are printed below, grouped by category.

In [ ]:
wrong = df[df["label"] == "WRONG"].copy()

print(f"Total wrong answers: {len(wrong)}\n")
print("Wrong answers by category:")
wrong_by_cat = wrong.groupby("category").size().rename("count").reset_index()
wrong_by_cat["category_name"] = wrong_by_cat["category"].map(CATEGORY_NAMES)
print(wrong_by_cat[["category", "category_name", "count"]].to_string(index=False))

print("\n" + "=" * 80)
print("Sample wrong answers (first 10):\n")
for _, row in wrong.head(10).iterrows():
    print(f"  Q: {row['question']}")
    print(f"  Gold:      {row['gold_answer']}")
    print(f"  Generated: {row['generated_answer'][:120]}")
    print(f"  Category:  {CATEGORY_NAMES.get(row['category'], '?')}")
    print(f"  Conv:      {row['conv_id']}")
    print()

## 8 · Context token analysis — Memori vs full-session baseline

How much context does the model actually need? This section compares two approaches:

| Approach | What goes into the prompt | Expected token count |
|----------|---------------------------|----------------------|
| **Memori (RAG)** | Only the top-K retrieved memories and summaries | Small, focused |
| **Baseline** | The entire conversation history for that user | Very large |

Lower token counts mean **less noise**, **lower latency**, and **lower API cost** — while (ideally) maintaining the same accuracy. The tables below show the reduction per conversation and per question category.


In [ ]:
import tiktoken

enc = tiktoken.encoding_for_model("gpt-4o-mini")
prompt_overhead = len(enc.encode(ANSWER_PROMPT))

# ── RAG context tokens per question ──────────────────────────────────
rag_tokens_per_q: list[dict] = []
for res in eval_results:
    mem_text = res.get("memories_text", "")
    n_tok = len(enc.encode(mem_text)) if mem_text else 0
    rag_tokens_per_q.append(
        {
            "conv_id": res["conv_id"],
            "category": res["category"],
            "context_tokens": n_tok,
            "question": res["question"],
        }
    )

tok_values = [r["context_tokens"] for r in rag_tokens_per_q]
avg_rag = sum(tok_values) / len(tok_values)
p50_rag = sorted(tok_values)[len(tok_values) // 2]

# ── Full-session baseline: conv_id → (total_tokens, n_sessions) ────────
# Same corpus as §1 (avoid a second download / disk read).
locomo_raw = locomo_data

conv_session_tokens: dict[str, tuple[int, int]] = {}
for conv in locomo_raw:
    cid = conv["sample_id"]
    conv_data = conv["conversation"]
    session_keys = sorted(
        [
            k
            for k in conv_data
            if k.startswith("session_") and not k.endswith("_date_time")
        ],
        key=lambda k: int(k.split("_")[1]),
    )
    all_text_parts: list[str] = []
    for skey in session_keys:
        for turn in conv_data[skey]:
            speaker = turn.get("speaker", "Unknown")
            content = turn.get("text", "")
            blip = turn.get("blip_caption", "")
            query = turn.get("query", "")
            if blip or query:
                img_desc = query if query else blip
                if query and blip:
                    img_desc = f"{query} — {blip}"
                content = f"[Shared image: {img_desc}] {content}".strip()
            all_text_parts.append(f"{speaker}: {content}")
    full_text = "\n".join(all_text_parts)
    conv_session_tokens[cid] = (len(enc.encode(full_text)), len(session_keys))

conv_ids_in_eval = {r["conv_id"] for r in rag_tokens_per_q}
avg_baseline = sum(conv_session_tokens[c][0] for c in conv_ids_in_eval) / len(
    conv_ids_in_eval
)

# ── Report ───────────────────────────────────────────────────────────
print("=" * 70)
print("  CONTEXT TOKEN ANALYSIS – MEMORI vs FULL-SESSION BASELINE")
print("=" * 70)
print(f"\n  Questions analysed:      {len(tok_values)}")
print(f"  Retrieved items / q:     {TOP_K}")
print(f"  ANSWER_PROMPT overhead:  {prompt_overhead} tokens (fixed)")

print("\n  Memori context tokens per question (memories + summaries):")
print(f"    Mean:   {avg_rag:>8,.0f}")
print(f"    Median: {p50_rag:>8,}")
print(f"    Min:    {min(tok_values):>8,}")
print(f"    Max:    {max(tok_values):>8,}")

print("\n  Full-session baseline (all sessions as naive context):")
for cid in sorted(conv_ids_in_eval):
    tok, ns = conv_session_tokens[cid]
    print(f"    {cid}: {tok:>8,} tokens  ({ns} sessions)")

print(f"\n  {'Metric':<30s} {'Memori':>10s} {'Baseline':>10s}")
print(f"  {'-' * 30} {'-' * 10} {'-' * 10}")
print(
    f"  {'Avg context tokens/question':<30s} {avg_rag:>10,.0f} {avg_baseline:>10,.0f}"
)
print(
    f"  {'+ prompt overhead':<30s} {avg_rag + prompt_overhead:>10,.0f} {avg_baseline + prompt_overhead:>10,.0f}"
)
print(f"  {'Ratio (Memori / baseline)':<30s} {avg_rag / avg_baseline:>10.2%}")
print(f"  {'Context reduction':<30s} {1 - avg_rag / avg_baseline:>10.1%}")

# ── Per-conversation breakdown ───────────────────────────────────────
print("\n  Per-conversation breakdown:")
print(
    f"  {'Conv':<10s} {'RAG avg':>10s} {'Baseline':>10s} {'Reduction':>10s} {'Questions':>10s}"
)
print(f"  {'-' * 10} {'-' * 10} {'-' * 10} {'-' * 10} {'-' * 10}")
for cid in sorted(conv_ids_in_eval):
    q_tokens = [r["context_tokens"] for r in rag_tokens_per_q if r["conv_id"] == cid]
    avg_q = sum(q_tokens) / len(q_tokens) if q_tokens else 0
    base_tok = conv_session_tokens[cid][0]
    reduction = 1 - avg_q / base_tok if base_tok else 0
    print(
        f"  {cid:<10s} {avg_q:>10,.0f} {base_tok:>10,} {reduction:>10.1%} {len(q_tokens):>10}"
    )

# ── Per-category breakdown ───────────────────────────────────────────
print("\n  Per-category breakdown:")
print(f"  {'Category':<15s} {'Memori avg':>10s} {'Questions':>10s}")
print(f"  {'-' * 15} {'-' * 10} {'-' * 10}")
for cat_id in sorted({r["category"] for r in rag_tokens_per_q}):
    q_tokens = [
        r["context_tokens"] for r in rag_tokens_per_q if r["category"] == cat_id
    ]
    avg_q = sum(q_tokens) / len(q_tokens) if q_tokens else 0
    cat_name = CATEGORY_NAMES.get(cat_id, f"Cat-{cat_id}")
    print(f"  {cat_name:<15s} {avg_q:>10,.0f} {len(q_tokens):>10}")

print("=" * 70)